# Домашнє завдання: Проектування розподіленої системи кешування (Відеоплатформа - 2BeSmart)

## Частина 1: Аналіз вимог та оцінка масштабу

### 1. Back-of-the-Envelope розрахунки
Для розрахунків зробимо базове припущення: з 100 млн щомісячних активних користувачів (MAU), щоденно активними є близько 50% (DAU = 50 млн). Це стандартна метрика для подібних платформ.

**А) Розрахунок розміру одного відео:**
* Тривалість: 5 хвилин = 300 секунд.
* Бітрейт (якість): 5 Мбіт/с.

$$\text{Обсяг (Мбіт)} = 300 \text{ с} \times 5 \text{ Мбіт/с} = 1500 \text{ Мбіт}$$
$$\text{Обсяг (МБ)} = \frac{1500 \text{ Мбіт}}{8 \text{ біт/байт}} = \mathbf{187.5 \text{ МБ}}$$

**Б) Очікуваний обсяг даних для зберігання (за 1 день):**
* **Відеофайли:** $$500,000 \text{ відео} \times 187.5 \text{ МБ} \approx 93,750,000 \text{ МБ} \approx \mathbf{93.75 \text{ ТБ/день}}$$
* **Метадані:** $$500,000 \text{ відео} \times 2 \text{ КБ} = 1,000,000 \text{ КБ} \approx \mathbf{1 \text{ ГБ/день}}$$
* **Коментарі:** $$500,000 \text{ відео} \times 10 \text{ ком.} \times 100 \text{ байт} = 500,000,000 \text{ байт} \approx \mathbf{500 \text{ МБ/день}}$$

*Річний приріст сховища (Storage):* $$\text{Storage}_{year} = 93.75 \text{ ТБ} \times 365 \approx \mathbf{34.2 \text{ Петабайти (ПБ) на рік}}$$
*(Примітка: Розрахунок наведено тільки для оригіналів відео, без урахування реплікації та різних форматів якості, які збільшать цю цифру у 3-4 рази).*

**В) Пропускна здатність мережі (Network Bandwidth):**
* Кількість переглядів: $50,000,000 \text{ (DAU)} \times 5 \text{ переглядів} = 250,000,000 \text{ переглядів/день}$
* Щоденний вихідний трафік (Egress): $250,000,000 \times 187.5 \text{ МБ} \approx \mathbf{46.8 \text{ ПБ/день}}$

Розрахунок необхідної пропускної здатності (в бітах на секунду):
$$\text{Bandwidth} = \frac{46.8 \times 10^{15} \text{ байт} \times 8 \text{ біт}}{86400 \text{ секунд}} \approx \mathbf{4.3 \text{ Tbps (Терабіт на секунду)}}$$
*(Цей колосальний обсяг трафіку прямо вказує на абсолютну необхідність використання глобальної CDN мережі).*

**Г) Необхідні обчислювальні ресурси (QPS - Queries Per Second):**
* **Read QPS (Перегляди):** $\frac{250,000,000}{86400} \approx \mathbf{2,893 \text{ запитів/сек}}$ (середнє). Пікове навантаження (Peak QPS) $\approx \mathbf{5,786 \text{ запитів/сек}}$.
* **Write QPS (Завантаження відео):** $\frac{500,000}{86400} \approx \mathbf{5.8 \text{ запитів/сек}}$ (це важкі I/O операції, що вимагають асинхронної обробки).
* **Write QPS (Коментарі):** $\frac{5,000,000}{86400} \approx \mathbf{58 \text{ запитів/сек}}$.

---

### 2. Аналіз нефункціональних вимог (CAP-теорема та PACELC)
Для відеоплатформи "2BeSmart" найкритичнішими вимогами є **Доступність (Availability)** та **Стійкість до розділення (Partition Tolerance)**. 

Згідно з **CAP-теоремою**, у розподіленій системі під час мережевого збою ми маємо обрати між доступністю та консистентністю (узгодженістю). Наша система однозначно обирає **AP (Availability + Partition Tolerance)** з підходом **Eventual Consistency** (узгодженість у кінцевому підсумку).

Більше того, застосовуючи теорему **PACELC**, ми визначаємо поведінку системи і в нормальному стані. Наш вибір — **PA/EL**:
* **PA** (у разі Partition обираємо Availability): Якщо зв'язок між дата-центрами обірветься, користувачі все одно повинні мати змогу дивитися відео, навіть якщо коментарі тимчасово не синхронізуються.
* **EL** (Else обираємо Latency): За нормальної роботи системи для нас важливіша мінімальна затримка (Latency), ніж абсолютна узгодженість (Consistency). Ми готові віддати клієнту відео з кешу, навіть якщо лічильник переглядів на ньому ще не оновився.

**Обґрунтування:**
1. **Бізнес-цінність:** Якщо користувач не зможе відкрити відео або сторінка завантажуватиметься довго (втрата доступності або висока латентність), він піде з платформи. Це пряма втрата грошей.
2. **Толерантність до затримок даних:** Якщо користувач залишив коментар, або кількість лайків оновиться для інших на 5 секунд пізніше — це абсолютно некритично для UX. Ніхто не помітить, що в одного користувача відео має 1 000 000 переглядів, а в іншого — 1 000 005. 
3. *Виняток:* Суворої консистентності (CP) вимагатиме лише вузька підсистема — фінансові транзакції (купівля підписки або донати авторам), яка буде ізольована в окремому реляційному домені.